## 0.1. Veri Kümesini Yükleme

Modeli 1100 Türkçe isim üzerinde eğiteceğiz. Her isim kısa bir belge
(document) olarak kabul edilecektir.

Veri kümesindeki isimler önce temizlenecek, küçük harfe dönüştürülecek
ve ardından eğitim sırasının etkisini azaltmak için karıştırılacaktır.

In [1]:
import random
import math
docs = [
    line.strip()
        .replace(" ", "")## aradaki boşlukları da sildim.
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
    for line in open("../data/raw_names.txt", encoding="utf-8")
        .read()
        .strip()
        .split("\n")
    if line.strip()
]

random.shuffle(docs)

In [2]:
# Listeyi bulunduğu yerde karıştırır
random.shuffle(docs)

print("Karıştırılmış ilk 10 isim:")
print(docs[:10])

Karıştırılmış ilk 10 isim:
['seyhan', 'kezban', 'buğlem', 'berk', 'durmuşali', 'abidin', 'miyase', 'mehmetakif', 'eslem', 'meltem']


## 0.2. Tokenizer

Dil modeli karakterlerle doğrudan çalışamaz. Bu nedenle her farklı
karaktere sayısal bir kimlik, yani token ID verilecektir.

Bu projede karakter düzeyinde tokenization kullanılacaktır.

In [3]:
uchars = sorted(set(''.join(docs)))

BOS = len(uchars)
vocab_size = len(uchars) + 1

print(uchars)
print("Karakter sayısı:", len(uchars))
print("BOS token ID:", BOS)
print("Vocabulary boyutu:", vocab_size)

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'y', 'z', 'ç', 'ö', 'ü', 'ğ', 'ı', 'ş']
Karakter sayısı: 29
BOS token ID: 29
Vocabulary boyutu: 30


## 0.3. Sayım Tablosu

Bigram modelimiz iki boyutlu bir sayım tablosudur.

- Satırlar mevcut tokenı temsil eder.
- Sütunlar bir sonraki tokenı temsil eder.
- Her hücre, bir tokenın ardından diğer tokenın kaç kez geldiğini tutar.

Başlangıçta bütün hücreler sıfırdır.

In [4]:
state_dict = [[0] * vocab_size for _ in range(vocab_size)]
print("Toplam hücre sayısı:", len(state_dict) * len(state_dict[0]))


Toplam hücre sayısı: 900


## 0.4. Bigram Modeli

Bigram modeli, mevcut tokenı kullanarak bir sonraki tokenın olasılıklarını
hesaplar.

Model yalnızca iki token arasındaki ilişkiye bakar:

- Mevcut token
- Bir sonraki token

Daha önce gelen tokenları hatırlamaz.

Laplace smoothing
Fonksiyonda iki ekleme bulunuyor.Böylece hiç görülmeyen bir geçişin olasılığı çok küçük olur ama tamamen sıfır olmaz.
Her hücreye 1 eklenmiş gibi davranıyoruz. Bizde bir satırda 30 hücre olduğu için toplam sayıya da 30 eklemeliyiz

In [5]:
def bigram(token_id):
    row = state_dict[token_id]
    total = sum(row) + vocab_size
    return [(c + 1) / total for c in row]

## 0.5. Modelin Eğitilmesi

Her eğitim adımında bir isim seçilir, tokenlara dönüştürülür ve başına
ve sonuna BOS tokenı eklenir.

Model önce ismi ne kadar iyi tahmin ettiğini loss ile ölçer. Ardından
isimdeki karakter geçişlerinin sayılarını bir artırır.

In [6]:
num_steps =1000
for step in range(num_steps):
    doc =docs[step% len(docs)]
    tokens =[BOS] + [uchars.index(ch)for ch in doc]+[BOS]
    n=len(tokens)-1

    losses =[]
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id],tokens[pos_id+1]
        probs =bigram(token_id)
        loss_t= -math.log(probs[target_id])
        losses.append(loss_t)
    loss = (1/n)*sum(losses)

    for pos_id in range(n):
        token_id,target_id =tokens[pos_id],tokens[pos_id+1]
        state_dict[token_id][target_id]+=1
        

In [7]:
print("Son işlenen isim:", doc)
print("Son loss:", loss)
print("Tablodaki toplam geçiş sayısı:", sum(sum(row) for row in state_dict))

Son işlenen isim: mehmet
Son loss: 2.139992003634347
Tablodaki toplam geçiş sayısı: 6708


## 0.6 Loss

Loss, modelin gerçek sonraki tokenı tahmin etmekte ne kadar başarılı
olduğunu ölçer.

Modelin doğru tokene verdiği olasılık yüksekse loss düşük, olasılık
düşükse loss yüksek olur.

In [8]:
example_probabilities = [1.0, 0.8, 0.5, 0.1, 0.01]

for probability in example_probabilities:
    example_loss = -math.log(probability)

    print(
        f"Olasılık: {probability:.2f} "
        f"-- Loss: {example_loss:.4f}"
    )

Olasılık: 1.00 -- Loss: -0.0000
Olasılık: 0.80 -- Loss: 0.2231
Olasılık: 0.50 -- Loss: 0.6931
Olasılık: 0.10 -- Loss: 2.3026
Olasılık: 0.01 -- Loss: 4.6052


## 0.7 Inference

Inference, eğitilmiş model kullanılarak yeni örnekler üretilmesi
işlemidir.

İsim üretimi BOS tokenıyla başlar. Model, mevcut tokenın ardından
gelebilecek bütün tokenlar için bir olasılık dağılımı üretir. Bu
dağılımdan rastgele bir sonraki token seçilir.

Seçilen token yeniden modele verilerek üretim devam eder. Model tekrar
BOS tokenını seçerse isim tamamlanır. BOS üretilmezse işlem en fazla
16 token boyunca sürdürülür.

Inference sırasında sayım tablosu güncellenmez; model yalnızca eğitim
sırasında öğrendiği olasılıkları kullanır.

In [9]:
for sample_idx in range(20):#isim sayısı
    token_id =BOS
    sample =[]

    for _ in range (16):
        token_id =random.choices(
            range (vocab_size),
            weights =bigram(token_id))[0]

        if token_id == BOS:
            break
        sample.append(uchars[token_id])

    print(f"sample {sample_idx +1:2d}:{''.join(sample)}")

sample  1:oaydizasa
sample  2:begüllpevamu
sample  3:ta
sample  4:etilebefemasan
sample  5:vçase
sample  6:osayn
sample  7:nğuçafetahimktiv
sample  8:süuminef
sample  9:n
sample 10:t
sample 11:mmttat
sample 12:hayen
sample 13:stl
sample 14:afaitupöderslliy
sample 15:başrkada
sample 16:fevğe
sample 17:n
sample 18:at
sample 19:n
sample 20:hsyjğdan
